In [ ]:
import os
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
from torchvision.models import densenet121, DenseNet121_Weights       # ← CHANGED

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import csv
import warnings
warnings.filterwarnings("ignore")


KVASIR_ROOT    = "/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset"
SYNTHETIC_ROOT = "/kaggle/input/datasets/yumna19/endoscopy-medical-image-generation/generated_images"
OUTPUT_ROOT    = "/kaggle/working/split_dataset"
RESUME_CKPT_DIR  = ""

IMG_SIZE          = 224
BATCH_SIZE        = 32
EPOCHS            = 90
SYNTH_TRAIN_RATIO = 0.90
LR                = 1e-4
WEIGHT_DECAY      = 1e-3
LABEL_SMOOTHING   = 0.03
N_SPLITS          = 5
TEST_SPLIT        = 0.15
RANDOM_SEED       = 42
NUM_WORKERS       = 2
PATIENCE          = 15

EXPERIMENTS     = ["unfiltered"]
FILTER_KEEP_PCT = 0.80
START_FOLD      = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

def collect_image_paths(root_dir):
    samples = []
    classes = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])
    print(f"\nClasses found in '{root_dir}': {classes}")
    for cls in classes:
        cls_dir = os.path.join(root_dir, cls)
        sub = os.listdir(cls_dir)
        if len(sub) == 1 and os.path.isdir(os.path.join(cls_dir, sub[0])):
            cls_dir = os.path.join(cls_dir, sub[0])
        for fname in sorted(os.listdir(cls_dir)):
            if Path(fname).suffix.lower() in VALID_EXTS:
                samples.append((os.path.join(cls_dir, fname), cls))
    print(f"Total images: {len(samples)}")
    return samples, classes


def copy_files(samples, split_name, base_dir):
    for src, cls in samples:
        dst_dir = os.path.join(base_dir, split_name, cls)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(src, dst_dir)


def create_splits(kvasir_root, synthetic_root, output_root):
    real_samples, real_classes = collect_image_paths(kvasir_root)
    real_trainval, real_test = train_test_split(
        real_samples, test_size=TEST_SPLIT,
        random_state=RANDOM_SEED,
        stratify=[s[1] for s in real_samples]
    )
    synth_samples, synth_classes = collect_image_paths(synthetic_root)
    assert set(real_classes) == set(synth_classes)
    synth_train = synth_samples

    print(f"\n── Data split ──")
    print(f"  Real train+val : {len(real_trainval)}")
    print(f"  Real test      : {len(real_test)}")
    print(f"  Synthetic train: {len(synth_train)}")

    copy_files(real_trainval, "real_train", output_root)
    copy_files(synth_train,   "synthetic",  output_root)
    copy_files(real_test,     "test",       output_root)
    return real_classes, len(real_trainval), len(synth_train)


_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])


MODEL_REGISTRY = {}

def _register(name):
    def decorator(fn):
        MODEL_REGISTRY[name] = fn
        return fn
    return decorator


@_register("densenet121")
def build_densenet121(num_classes):
    model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)   
    
    for p in model.parameters():
        p.requires_grad = False


    for name, module in model.features.named_children():
        if name in ("denseblock4", "norm5"):
            for p in module.parameters():
                p.requires_grad = True


    in_f = model.classifier.in_features   
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.50, inplace=False),
        nn.Linear(in_f, 512),
        nn.LeakyReLU(negative_slope=0.01, inplace=False),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.30, inplace=False),
        nn.Linear(512, num_classes),
    )

    _print_param_count(model, "DenseNet-121")
    return model.to(DEVICE)


def _print_param_count(model, name):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] Trainable: {trainable:>10,} / {total:>10,} "
          f"({100 * trainable / total:.1f}%)")



MODEL_INPUT_SIZES = {
    "densenet121": 224,   
}

MODEL_DISPLAY_NAMES = {
    "densenet121": "DenseNet-121",
}


def train_one_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = out.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        running_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(out, dim=1)
        _, preds = probs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
    all_probs = np.vstack(all_probs)
    return running_loss / total, correct / total, all_preds, all_labels, all_probs



def run_kfold_for_model(model_name, output_root, num_classes, class_names,
                        synth_mode="unfiltered"):
    img_size = MODEL_INPUT_SIZES[model_name]
    tf       = get_transform(img_size)

    real_train_ds = datasets.ImageFolder(
        os.path.join(output_root, "real_train"), transform=tf
    )

    if synth_mode == "filtered":
        synth_folder = "synthetic_filtered"
        synth_path   = os.path.join(output_root, synth_folder)
        if not os.path.isdir(synth_path):
            raise FileNotFoundError(f"Filtered folder '{synth_path}' not found.")
    else:
        synth_folder = "synthetic"
        synth_path   = os.path.join(output_root, synth_folder)

    synthetic_ds = datasets.ImageFolder(synth_path, transform=tf)
    assert real_train_ds.class_to_idx == synthetic_ds.class_to_idx

    indices = np.arange(len(real_train_ds))
    labels  = np.array(real_train_ds.targets)
    skf     = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
    fold_results = []

    exp_label = "Exp2: Real + Full Synthetic" if synth_mode == "unfiltered" \
                else f"Exp3: Real + Filtered Synthetic (top {int(FILTER_KEEP_PCT*100)}%)"

    print(f"\n{'='*65}")
    print(f"  EXPERIMENT : {exp_label}")
    print(f"  MODEL      : {MODEL_DISPLAY_NAMES[model_name]}  "
          f"(input {img_size}×{img_size})")
    print(f"  Real train : {len(real_train_ds)}  |  "
          f"Synthetic ({synth_folder}): {len(synthetic_ds)}  |  Folds: {N_SPLITS}")
    print(f"{'='*65}")

    ckpt_dir = f"checkpoints/{synth_mode}"
    os.makedirs(ckpt_dir, exist_ok=True)
    if START_FOLD > 1 and RESUME_CKPT_DIR:
        for f in range(1, START_FOLD):
            src = os.path.join(RESUME_CKPT_DIR, f"{model_name}_fold{f}.pth")
            dst = os.path.join(ckpt_dir, f"{model_name}_fold{f}.pth")
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Copied existing checkpoint: fold {f}")
            elif not os.path.exists(src):
                print(f"  WARNING: checkpoint not found: {src}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels), 1):

        if fold < START_FOLD:
            print(f"\n  ── Fold {fold}/{N_SPLITS} — SKIPPED (already done) ──")
            fold_results.append({
                "fold": fold, "best_val_acc": None,
                "history": {"train_loss":[], "train_acc":[],
                             "val_loss":[],   "val_acc":[]},
            })
            continue

        print(f"\n  ── Fold {fold}/{N_SPLITS} ──")

        real_len      = len(train_idx)
        synth_keep    = int(len(synthetic_ds) * 0.90)
        synth_indices = np.random.choice(len(synthetic_ds), synth_keep, replace=False)
        synth_subset  = Subset(synthetic_ds, synth_indices)
        train_combined = ConcatDataset([Subset(real_train_ds, train_idx), synth_subset])
        val_subset     = Subset(real_train_ds, val_idx)

        print(f"     Train fold : {real_len}  +  {synth_keep} synthetic = "
              f"{len(train_combined)} total")
        print(f"     Val fold   : {len(val_idx)} (real only)")

        train_loader = DataLoader(train_combined, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        val_loader   = DataLoader(val_subset,     batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        model = MODEL_REGISTRY[model_name](num_classes)

        label_arr    = np.array(real_train_ds.targets)[train_idx]
        class_counts = np.bincount(label_arr, minlength=num_classes).astype(float)
        class_weights = 1.0 / (class_counts + 1e-6)

        hard_class_names = ["esophagitis", "normal-z-line",
                            "dyed-lifted-polyps", "dyed-resection-margins"]
        for cls_name in hard_class_names:
            if cls_name in real_train_ds.class_to_idx:
                idx = real_train_ds.class_to_idx[cls_name]
                class_weights[idx] *= 1.5

        class_weights = class_weights / class_weights.sum() * num_classes
        weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weight_tensor,
                                        label_smoothing=LABEL_SMOOTHING)

 # CHANGED: now only 2 unfrozen backbone groups instead of 3
        late_block_params = []   # denseblock4 + norm5
        head_params       = []   # classifier

        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if "classifier" in name:
                head_params.append(p)
            else:
                late_block_params.append(p)   # everything else unfrozen = denseblock4 + norm5

        optimizer = optim.AdamW([
            {"params": late_block_params, "lr": LR * 0.10, "weight_decay": WEIGHT_DECAY * 2.0},
            {"params": head_params,       "lr": LR,        "weight_decay": WEIGHT_DECAY * 4.0},
        ])

        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[LR * 0.10, LR],
            steps_per_epoch=len(train_loader),
            epochs=EPOCHS,
            pct_start=0.30,
            anneal_strategy='cos',
            div_factor=10.0,
            final_div_factor=500.0,
        )

        best_val_acc, best_state = 0.0, None
        patience_ctr = 0
        history = {k: [] for k in ["train_loss","train_acc","val_loss","val_acc"]}

        for epoch in range(1, EPOCHS + 1):
            t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, scheduler, criterion)
            v_loss, v_acc, _, _, _ = evaluate(model, val_loader, criterion)
            history["train_loss"].append(t_loss)
            history["train_acc"].append(t_acc)
            history["val_loss"].append(v_loss)
            history["val_acc"].append(v_acc)

            print(f"     Epoch {epoch:02d}/{EPOCHS}  "
                  f"Train {t_acc:.4f} ({t_loss:.4f})  |  "
                  f"Val {v_acc:.4f} ({v_loss:.4f})")

            if v_acc > best_val_acc:
                best_val_acc = v_acc
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"     Early stop at epoch {epoch}")
                    break

        torch.save(best_state, f"{ckpt_dir}/{model_name}_fold{fold}.pth")
        fold_results.append({"fold": fold, "best_val_acc": best_val_acc, "history": history})
        print(f"\n  ✓ Fold {fold}  best val acc: {best_val_acc*100:.2f}%")

    return fold_results



@torch.no_grad()
def _collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs  = imgs.to(DEVICE)
        out   = model(imgs)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.extend(labels.numpy())
    return np.vstack(all_probs), np.array(all_labels)


def evaluate_model_on_test(model_name, output_root, num_classes, class_names,
                            synth_mode="unfiltered"):
    img_size    = MODEL_INPUT_SIZES[model_name]
    tf          = get_transform(img_size)
    test_ds     = datasets.ImageFolder(
        os.path.join(output_root, "test"), transform=tf
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )
 
    prob_sum            = None
    labels              = None
    ckpt_dir            = f"checkpoints/{synth_mode}"
    per_fold_test_accs  = []          
 
    for fold in range(1, N_SPLITS + 1):
        ckpt  = f"{ckpt_dir}/{model_name}_fold{fold}.pth"
        model = MODEL_REGISTRY[model_name](num_classes)
        checkpoint = torch.load(ckpt, map_location=DEVICE)
        state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
        model.load_state_dict(state_dict)
 
        probs, lbls = _collect_probs(model, test_loader)
        prob_sum = probs if prob_sum is None else prob_sum + probs
        labels   = lbls
 
        fold_acc = accuracy_score(lbls, probs.argmax(1))
        per_fold_test_accs.append(fold_acc)          # ← NEW
        print(f"  [{MODEL_DISPLAY_NAMES[model_name]}] [{synth_mode}] "
              f"Fold {fold} test acc: {fold_acc*100:.2f}%")
 
    avg_probs   = prob_sum / N_SPLITS
    final_preds = avg_probs.argmax(axis=1)
    acc = accuracy_score(labels, final_preds)
    f1  = f1_score(labels, final_preds, average="weighted")
    try:
        auc = roc_auc_score(labels, avg_probs, multi_class="ovr",
                            average="weighted")
    except Exception:
        auc = float("nan")
 
    print(f"\n  [{MODEL_DISPLAY_NAMES[model_name]}] [{synth_mode}] "
          f"Soft-Ensemble  Acc={acc*100:.2f}%  F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(labels, final_preds, target_names=class_names))
 
    _plot_confusion_matrix(
        labels, final_preds, class_names,
        title=f"{MODEL_DISPLAY_NAMES[model_name]} [{synth_mode}]  —  Acc={acc*100:.2f}%",
        filename=f"cm_{model_name}_{synth_mode}.png"
    )
 
    return {
        "model":               model_name,
        "synth_mode":          synth_mode,
        "accuracy":            acc,
        "f1_weighted":         f1,
        "auc_ovr":             auc,
        "preds":               final_preds,
        "labels":              labels,
        "per_fold_test_accs":  per_fold_test_accs,  
    }


def _plot_confusion_matrix(labels, preds, class_names, title, filename, cmap="Blues"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(max(8, len(class_names)), 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"  Saved → {filename}")


def plot_training_curves(fold_results, model_name, synth_mode):
    display = MODEL_DISPLAY_NAMES[model_name]
    fig, axes = plt.subplots(2, N_SPLITS, figsize=(5 * N_SPLITS, 8))
    for i, fr in enumerate(fold_results):
        h = fr["history"]
        axes[0, i].plot(h["train_loss"], label="Train")
        axes[0, i].plot(h["val_loss"],   label="Val")
        axes[0, i].set_title(f"Fold {fr['fold']} – Loss")
        axes[0, i].legend()
        axes[1, i].plot(h["train_acc"], label="Train")
        axes[1, i].plot(h["val_acc"],   label="Val")
        axes[1, i].set_title(f"Fold {fr['fold']} – Acc")
        axes[1, i].legend()
    plt.suptitle(f"{display} [{synth_mode}] — K-Fold Training Curves", fontsize=13)
    plt.tight_layout()
    fname = f"curves_{model_name}_{synth_mode}.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"  Saved → {fname}")



def save_all_metrics(fold_results, result):
    model_name = result["model"]
    mode       = result["synth_mode"]
    suffix     = f"_{mode}"
 

    with open(f"training_log{suffix}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["fold", "epoch",
                    "train_loss", "train_acc",
                    "val_loss",   "val_acc"])
        for fr in fold_results:
            h = fr["history"]
            for ep in range(len(h["train_loss"])):
                w.writerow([
                    fr["fold"], ep + 1,
                    round(h["train_loss"][ep], 4),
                    round(h["train_acc"][ep],  4),
                    round(h["val_loss"][ep],   4),
                    round(h["val_acc"][ep],    4),
                ])
    print(f"Saved → training_log{suffix}.csv")
 

    ensemble_test_acc = result["accuracy"]
    per_fold_test     = result.get("per_fold_test_accs", [None] * N_SPLITS)
 
    with open(f"fold_summary{suffix}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "fold",
            "best_val_acc",        
            "fold_test_acc",       
            "ensemble_test_acc",   
            "f1_weighted",        
            "auc_ovr",           
        ])
        for fr in fold_results:
            fold_idx     = fr["fold"] - 1
            fold_test    = (round(per_fold_test[fold_idx], 4)
                            if fr["best_val_acc"] is not None
                            and fold_idx < len(per_fold_test)
                            else "N/A")
            best_val     = (round(fr["best_val_acc"], 4)
                            if fr["best_val_acc"] is not None else "N/A")
            w.writerow([
                fr["fold"],
                best_val,
                fold_test,
                round(ensemble_test_acc, 4),
                round(result["f1_weighted"], 4),
                round(result["auc_ovr"], 4)
                    if not (isinstance(result["auc_ovr"], float)
                            and result["auc_ovr"] != result["auc_ovr"])  # nan check
                    else "N/A",
            ])
 
        # ── summary rows at the bottom ────────────────────────────
        valid_vals  = [fr["best_val_acc"] for fr in fold_results
                       if fr["best_val_acc"] is not None]
        valid_tests = [v for v in per_fold_test if v is not None]
        w.writerow([])   # blank separator
        w.writerow(["MEAN",
                    round(float(np.mean(valid_vals)),  4),
                    round(float(np.mean(valid_tests)), 4),
                    round(ensemble_test_acc, 4),
                    round(result["f1_weighted"], 4),
                    ""])
        w.writerow(["STD",
                    round(float(np.std(valid_vals)),  4),
                    round(float(np.std(valid_tests)), 4),
                    "", "", ""])
 
    print(f"Saved → fold_summary{suffix}.csv")
 
    valid_accs = [fr["best_val_acc"] for fr in fold_results
                  if fr["best_val_acc"] is not None]
    summary = {
        "model":                MODEL_DISPLAY_NAMES[model_name],
        "synth_mode":           mode,
        "ensemble_test_acc":    float(result["accuracy"]),
        "f1_weighted":          float(result["f1_weighted"]),
        "auc_ovr":              (float(result["auc_ovr"])
                                 if not np.isnan(result["auc_ovr"]) else None),
        "per_fold_test_accs":   [round(v, 4) for v in per_fold_test],
        "kfold_val_accs":       [fr["best_val_acc"] for fr in fold_results],
        "kfold_val_mean":       float(np.mean(valid_accs)),
        "kfold_val_std":        float(np.std(valid_accs)),
    }
    with open(f"results_summary{suffix}.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"Saved → results_summary{suffix}.json")
 
 
def save_experiment_comparison(all_fold_results, all_results):
    rows = []
    for fr_list, res in zip(all_fold_results, all_results):
        mean_train = np.mean([fr["history"]["train_acc"][-1] for fr in fr_list if fr["history"]["train_acc"]])
        mean_val   = np.mean([fr["best_val_acc"] for fr in fr_list if fr["best_val_acc"] is not None])
        std_val    = np.std( [fr["best_val_acc"] for fr in fr_list if fr["best_val_acc"] is not None])
        mode       = res["synth_mode"]
        exp_label  = "Exp2: Real + Full Synthetic" if mode == "unfiltered" \
                     else f"Exp3: Real + Filtered ({int(FILTER_KEEP_PCT*100)}%)"
        rows.append({
            "experiment":     exp_label,
            "synth_mode":     mode,
            "mean_train_acc": round(float(mean_train), 4),
            "mean_val_acc":   round(float(mean_val),   4),
            "std_val_acc":    round(float(std_val),    4),
            "test_acc":       round(float(res["accuracy"]),    4),
            "f1_weighted":    round(float(res["f1_weighted"]), 4),
            "auc_ovr":        round(float(res["auc_ovr"]), 4)
                              if not np.isnan(res["auc_ovr"]) else "N/A",
        })
    with open("experiment_comparison.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)
    with open("experiment_comparison.json", "w") as f:
        json.dump(rows, f, indent=2)

    print("\n" + "="*75)
    print("  EXPERIMENT COMPARISON")
    print("="*75)
    print(f"  {'Experiment':<38} {'Val Acc':>10} {'Test Acc':>10} {'F1':>8} {'AUC':>8}")
    print(f"  {'-'*74}")
    for r in rows:
        print(f"  {r['experiment']:<38} "
              f"{r['mean_val_acc']*100:>9.2f}% "
              f"{r['test_acc']*100:>9.2f}% "
              f"{r['f1_weighted']:>8.4f} "
              f"{str(r['auc_ovr']):>8}")
    print("="*75)

 
import cv2
from PIL import Image

def compute_image_quality_score(img_path):
    try:
        img_pil = Image.open(img_path).convert("RGB").resize((224, 224))
        img_np  = np.array(img_pil)
        gray      = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
        contrast  = img_np.std()
        hsv        = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
        saturation = hsv[:, :, 1].mean()
        return {"path": img_path, "sharpness": sharpness,
                "contrast": contrast, "saturation": saturation}
    except Exception as e:
        print(f"  Warning: could not score {img_path}: {e}")
        return None


def filter_synthetic_by_quality(synthetic_root, output_root, class_names, keep_top_pct=0.80):
    print(f"\n── Synthetic quality filter (keeping top {keep_top_pct*100:.0f}%) ──")
    total_kept = total_dropped = 0
    for cls in class_names:
        cls_dir = os.path.join(synthetic_root, cls)
        if not os.path.isdir(cls_dir):
            continue
        sub = os.listdir(cls_dir)
        if len(sub) == 1 and os.path.isdir(os.path.join(cls_dir, sub[0])):
            cls_dir = os.path.join(cls_dir, sub[0])
        img_paths = [os.path.join(cls_dir, f) for f in sorted(os.listdir(cls_dir))
                     if Path(f).suffix.lower() in VALID_EXTS]
        scores = [s for s in (compute_image_quality_score(p) for p in img_paths) if s is not None]
        if not scores:
            continue
        def norm(arr):
            r = arr.max() - arr.min()
            return (arr - arr.min()) / r if r > 0 else np.zeros_like(arr)
        composite = (norm(np.array([s["sharpness"]  for s in scores])) +
                     norm(np.array([s["contrast"]   for s in scores])) +
                     norm(np.array([s["saturation"] for s in scores]))) / 3.0
        n_keep  = max(1, int(len(scores) * keep_top_pct))
        top_idx = np.argsort(composite)[::-1][:n_keep]
        kept    = [scores[i]["path"] for i in top_idx]
        dropped = len(scores) - n_keep
        dst_dir = os.path.join(output_root, "synthetic_filtered", cls)
        os.makedirs(dst_dir, exist_ok=True)
        for src in kept:
            shutil.copy2(src, dst_dir)
        total_kept += n_keep; total_dropped += dropped
        print(f"  [{cls}] {len(scores)} → kept {n_keep}, dropped {dropped}")
    print(f"\n  Total kept: {total_kept}  |  Total dropped: {total_dropped}")
    return os.path.join(output_root, "synthetic_filtered")


 
if __name__ == "__main__":

    print("\n── STEP 1: Creating splits ──")
    classes, n_real, n_synth = create_splits(KVASIR_ROOT, SYNTHETIC_ROOT, OUTPUT_ROOT)
    num_classes = len(classes)
    print(f"\nClasses ({num_classes}): {classes}")

    MODEL_NAME       = "densenet121"   
    all_fold_results = []
    all_results      = []

    for exp_mode in EXPERIMENTS:
        exp_label = ("Exp2: Real + Full Synthetic" if exp_mode == "unfiltered"
                     else f"Exp3: Real + Filtered Synthetic (top {int(FILTER_KEEP_PCT*100)}%)")
        print(f"\n\n{'#'*65}")
        print(f"  RUNNING {exp_label}")
        print(f"{'#'*65}")

        fold_results = run_kfold_for_model(
            MODEL_NAME, OUTPUT_ROOT, num_classes, classes, synth_mode=exp_mode
        )
        kf_accs = [fr["best_val_acc"] for fr in fold_results if fr["best_val_acc"] is not None]
        print(f"\n  [{MODEL_DISPLAY_NAMES[MODEL_NAME]}] [{exp_mode}] "
              f"K-Fold Val: mean={np.mean(kf_accs)*100:.2f}%  std={np.std(kf_accs)*100:.2f}%")

        result = evaluate_model_on_test(
            MODEL_NAME, OUTPUT_ROOT, num_classes, classes, synth_mode=exp_mode
        )
        plot_training_curves(fold_results, MODEL_NAME, exp_mode)
        save_all_metrics(fold_results, result)
        all_fold_results.append(fold_results)
        all_results.append(result)

    print("\n── STEP 3: Saving experiment comparison ──")
    save_experiment_comparison(all_fold_results, all_results)